# E-commerce Orders: Data Cleaning, EDA and Outlier Analysis

Dataset: *Dataset for Data Analytics.xlsx* (DecodeLabs, Kaggle). Update `FILE_PATH` in the first code cell to point to your local copy before running.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

FILE_PATH = "/kaggle/input/datasets/akinmadehaisha/decodelabs/Dataset for Data Analytics.xlsx"  # paste your path

try:
    df_raw = pd.read_excel(FILE_PATH)          # untouched original
    print("Loaded successfully")
except FileNotFoundError:
    print("File not found - check FILE_PATH")

In [ ]:
print("Rows, columns:", df_raw.shape)
print("\nColumns:", list(df_raw.columns))
display(df_raw.head())
display(df_raw.tail())

In [ ]:
print(df_raw.dtypes, "\n")

missing = df_raw.isna().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
display(pd.DataFrame({"missing": missing, "percent": missing_pct})[missing > 0])

print("Fully duplicated rows:", df_raw.duplicated().sum())
print("Duplicate OrderIDs:", df_raw["OrderID"].duplicated().sum())
print("Duplicate TrackingNumbers:", df_raw["TrackingNumber"].duplicated().sum())

In [ ]:
# Leading/trailing spaces in text columns
text_cols = df_raw.select_dtypes(include="object").columns
for col in text_cols:
    n = (df_raw[col].dropna() != df_raw[col].dropna().str.strip()).sum()
    if n: print(f"{col}: {n} values with extra spaces")

# Category consistency
for col in ["Product", "PaymentMethod", "OrderStatus", "CouponCode", "ReferralSource"]:
    print(f"\n{col}:\n", df_raw[col].value_counts(dropna=False))

# Validity checks
print("\nDate range:", df_raw["Date"].min(), "to", df_raw["Date"].max())
print("Quantity <= 0:", (df_raw["Quantity"] <= 0).sum())
print("UnitPrice <= 0:", (df_raw["UnitPrice"] <= 0).sum())
mismatch = (abs(df_raw["Quantity"] * df_raw["UnitPrice"] - df_raw["TotalPrice"]) > 0.01).sum()
print("TotalPrice != Quantity x UnitPrice:", mismatch)

display(df_raw.describe(include="all").T)

In [ ]:
# Original stays untouched in df_raw; all changes happen in df_clean
df_clean = df_raw.copy()
cleaning_log = []   # we record every operation here

def log_step(step, affected, decision):
    cleaning_log.append({"Step": step, "Records affected": int(affected), "Decision": decision})

In [ ]:
# 1. Leading/trailing spaces in text columns
text_cols = df_clean.select_dtypes(include=["object", "string"]).columns
spaces_fixed = 0
for col in text_cols:
    stripped = df_clean[col].str.strip()
    spaces_fixed += (df_clean[col].notna() & (df_clean[col] != stripped)).sum()
    df_clean[col] = stripped
log_step("Strip spaces in text columns", spaces_fixed, "Applied as a safeguard")

# 2. Date column: force proper datetime, count anything unreadable
df_clean["Date"] = pd.to_datetime(df_clean["Date"], errors="coerce")
log_step("Invalid dates", df_clean["Date"].isna().sum(), "None found, nothing changed")

# 3. Duplicates
log_step("Duplicate rows", df_clean.duplicated().sum(), "None found, nothing removed")

# 4. Impossible numeric values
log_step("Quantity <= 0", (df_clean["Quantity"] <= 0).sum(), "None found")
log_step("UnitPrice <= 0", (df_clean["UnitPrice"]<= 0).sum(), "None found")
display(pd.DataFrame(cleaning_log))

In [ ]:
# 5. Missing CouponCode means "no coupon used", so label it instead of deleting or filling with 0
coupon_missing = df_clean["CouponCode"].isna().sum()
df_clean["CouponCode"] = df_clean["CouponCode"].fillna("No Coupon")
log_step("CouponCode blank -> 'No Coupon'", coupon_missing, "Kept rows; blank is meaningful")

# 6. Helper columns for later analysis (original columns unchanged)
df_clean["Year"] = df_clean["Date"].dt.year
df_clean["Month"] = df_clean["Date"].dt.month
df_clean["YearMonth"] = df_clean["Date"].dt.to_period("M").astype(str)
df_clean["HasCoupon"] = df_clean["CouponCode"] != "No Coupon"
df_clean["IsCompleted"] = df_clean["OrderStatus"].isin(["Shipped", "Delivered"])
log_step("Helper columns added (Year, Month, YearMonth, HasCoupon, IsCompleted)", len(df_clean), "New columns only; no data removed")

display(pd.DataFrame(cleaning_log))

In [ ]:
num_cols = ["Quantity", "UnitPrice", "ItemsInCart", "TotalPrice"]

stats = df_clean[num_cols].agg(["count", "mean", "median", "min", "max", "std", "var"]).T
stats["mode"] = [df_clean[c].mode()[0] for c in num_cols]
stats["range"] = stats["max"] - stats["min"]
stats["Q1 (25%)"] = df_clean[num_cols].quantile(0.25)
stats["Q2 (50%)"] = df_clean[num_cols].quantile(0.50)
stats["Q3 (75%)"] = df_clean[num_cols].quantile(0.75)
stats["IQR"] = stats["Q3 (75%)"] - stats["Q1 (25%)"]
stats["skewness"] = df_clean[num_cols].skew()

display(stats.round(2))

In [ ]:
for col in ["Product", "OrderStatus", "PaymentMethod", "ReferralSource", "CouponCode"]:
    display(df_clean[col].value_counts().to_frame("Orders").assign(
        Percent=lambda x: (x["Orders"] / len(df_clean) * 100).round(1)))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(df_clean[col], kde=True, bins=25, ax=ax, color="#2a6f97")
    ax.axvline(df_clean[col].mean(), color="red", linestyle="--", label="Mean")
    ax.axvline(df_clean[col].median(), color="green", linestyle="-", label="Median")
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Number of orders")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, col in zip(axes, num_cols):
    sns.boxplot(y=df_clean[col], ax=ax, color="#a9d6e5")
    ax.set_title(f"Boxplot of {col}")
    ax.set_ylabel(col)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, col in zip(axes, ["Quantity", "ItemsInCart"]):
    sns.countplot(x=df_clean[col], ax=ax, color="#2a6f97")
    ax.set_title(f"Orders by {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Number of orders")
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for col in num_cols:
    q1, q3 = df_clean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    iqr_out = df_clean[(df_clean[col] < lower) | (df_clean[col] > upper)]

    z = (df_clean[col] - df_clean[col].mean()) / df_clean[col].std()
    z_out = (z.abs() > 3).sum()

    rows.append({"Column": col, "Q1": q1, "Q3": q3, "IQR": iqr,
                 "Lower bound": lower, "Upper bound": upper,
                 "IQR outliers": len(iqr_out),
                 "IQR outlier %": round(len(iqr_out) / len(df_clean) * 100, 2),
                 "Z-score outliers (|z|>3)": z_out})

outlier_table = pd.DataFrame(rows).round(2)
display(outlier_table)

In [ ]:
q1, q3 = df_clean["TotalPrice"].quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
tp_outliers = df_clean[df_clean["TotalPrice"] > upper]

display(tp_outliers[["OrderID", "Product", "Quantity", "UnitPrice", "TotalPrice", "OrderStatus"]]
        .sort_values("TotalPrice", ascending=False))
print("Quantity values in outliers:", tp_outliers["Quantity"].unique())
print("Lowest UnitPrice among them:", tp_outliers["UnitPrice"].min())

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(x=df_clean["TotalPrice"], color="#a9d6e5")
sns.stripplot(x=tp_outliers["TotalPrice"], color="red", size=7, label="IQR outliers")
plt.title("TotalPrice Boxplot with Outliers Highlighted")
plt.xlabel("TotalPrice")
plt.legend()
plt.show()

In [ ]:
status_summary = df_clean.groupby("OrderStatus")["TotalPrice"].agg(["count", "sum", "mean"]).round(0)
display(status_summary)

plt.figure(figsize=(8, 5))
sns.barplot(x=status_summary.index, y=status_summary["sum"], color="#2a6f97")
plt.title("Gross Order Value by Order Status")
plt.xlabel("Order Status")
plt.ylabel("Total order value")
plt.show()

In [ ]:
completed = df_clean[df_clean["IsCompleted"]]
monthly = completed.groupby("YearMonth")["TotalPrice"].sum()

plt.figure(figsize=(14, 5))
plt.plot(monthly.index, monthly.values, marker="o", color="#2a6f97")
plt.xticks(rotation=90)
plt.title("Monthly Revenue from Completed Orders (Jan 2023 - Jun 2025)")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()
print("Peak month:", monthly.idxmax(), round(monthly.max(), 0))
print("Lowest month:", monthly.idxmin(), round(monthly.min(), 0))
display(completed.groupby("Year")["TotalPrice"].agg(["count", "sum"]).round(0))

In [ ]:
# Rebuild cleaned data in one go
df_clean = df_raw.copy()

# Strip spaces in text columns
for col in df_clean.select_dtypes(include=["object", "string"]).columns:
    df_clean[col] = df_clean[col].str.strip()

# Dates and coupon labels
df_clean["Date"] = pd.to_datetime(df_clean["Date"], errors="coerce")
df_clean["CouponCode"] = df_clean["CouponCode"].fillna("No Coupon")

# Helper columns
df_clean["Year"] = df_clean["Date"].dt.year
df_clean["Month"] = df_clean["Date"].dt.month
df_clean["YearMonth"] = df_clean["Date"].dt.to_period("M").astype(str)
df_clean["HasCoupon"] = df_clean["CouponCode"] != "No Coupon"
df_clean["IsCompleted"] = df_clean["OrderStatus"].isin(["Shipped", "Delivered"])

print(df_clean.shape)   # should be (1200, 19)

In [ ]:
completed_rate = {}
for col in ["Product", "PaymentMethod", "ReferralSource"]:
    rates = (pd.crosstab(df_clean[col], df_clean["OrderStatus"], normalize="index") * 100).round(1)
    rates["Cancelled+Returned %"] = rates["Cancelled"] + rates["Returned"]
    display(rates.sort_values("Cancelled+Returned %", ascending=False))

In [ ]:
corr = df_clean[num_cols].corr(method="pearson")
display(corr.round(3))

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Pearson Correlation Matrix of Numerical Variables")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

sns.scatterplot(data=df_clean, x="UnitPrice", y="TotalPrice", alpha=0.4, ax=axes[0], color="#2a6f97")
axes[0].set_title("UnitPrice vs TotalPrice")
axes[0].set_xlabel("Unit price")
axes[0].set_ylabel("Total price")

sns.stripplot(data=df_clean, x="Quantity", y="TotalPrice", alpha=0.3, ax=axes[1], color="#2a6f97")
axes[1].set_title("Quantity vs TotalPrice")
axes[1].set_xlabel("Quantity")
axes[1].set_ylabel("Total price")

sns.boxplot(data=df_clean, x="Quantity", y="ItemsInCart", ax=axes[2], color="#a9d6e5")
axes[2].set_title("Items in Cart by Quantity Ordered")
axes[2].set_xlabel("Quantity")
axes[2].set_ylabel("Items in cart")

plt.tight_layout()
plt.show()

In [ ]:
df_clean["DateNumber"] = df_clean["Date"].map(pd.Timestamp.toordinal)
print("Date vs TotalPrice:", round(df_clean["DateNumber"].corr(df_clean["TotalPrice"]), 3))
print("Month vs TotalPrice:", round(df_clean["Month"].corr(df_clean["TotalPrice"]), 3))

In [ ]:
status_pct = (pd.crosstab(df_clean["Product"], df_clean["OrderStatus"], normalize="index") * 100)
status_pct = status_pct.sort_values("Cancelled", ascending=False)

status_pct.plot(kind="bar", stacked=True, figsize=(11, 6), colormap="viridis")
plt.title("Order Status Mix by Product (% of orders)")
plt.xlabel("Product")
plt.ylabel("Percentage of orders")
plt.xticks(rotation=30)
plt.legend(title="Order status", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
df_clean.drop(columns=["DateNumber"], errors="ignore").to_csv("cleaned_orders.csv", index=False)
outlier_table.to_csv("outlier_summary.csv", index=False)
print("Saved.")

In [ ]:
from scipy.stats import chi2_contingency

df_clean["CancelledOrReturned"] = df_clean["OrderStatus"].isin(["Cancelled", "Returned"])
for col in ["Product", "PaymentMethod", "ReferralSource", "CouponCode"]:
    p1 = chi2_contingency(pd.crosstab(df_clean[col], df_clean["CancelledOrReturned"]))[1]
    p2 = chi2_contingency(pd.crosstab(df_clean[col], df_clean["OrderStatus"]))[1]
    print(f"{col}: cancel/return p = {p1:.3f} | full status mix p = {p2:.3f}")